# 02 — Data Scale Comparison: Chamfer vs Power-SM3

## Hypothesis

With more training data the model can generalise better, but the two losses may scale differently:

- **Chamfer loss** averages over all predicted→GT and GT→predicted nearest-neighbour distances. It is dominated by the overall point-cloud shape and may saturate once the model has seen enough typical scenes.
- **Power-SM3 (τ=0.12)** up-weights hard, under-covered ground-truth objects via the softmin temperature and the cubic power. Additional data provides more rare configurations, directly improving coverage — so Power-SM3 is expected to benefit *more* from data scale than Chamfer does.

**Experiment**: train both losses for n_train ∈ {1000, 5000, 20000, 70000} samples and track how validation loss, coverage, and CD evolve with dataset size.

In [1]:
import sys
sys.path.insert(0, '.')
import clevr_utils as cu
from influencerformer.losses import PowerSoftMinLoss, ChamferLoss
print(f'Device: {cu.DEVICE}')

Device: cuda


In [4]:
N_TRAIN_LIST = [5000, 10000, 20000]#, 70000]
N_EPOCHS = 50
SNAPSHOT_EPOCHS = {0, 5, 15, 35, 49}

In [ ]:
all_results = {}
val_loader = cu.make_dataloader('val', max_samples=1000)

for n_train in N_TRAIN_LIST:
    train_loader = cu.make_dataloader('train', max_samples=n_train)

    # --- Chamfer ---
    chamfer_label = f'Chamfer n={n_train}'
    print(f'\n=== Training {chamfer_label} ===')
    metrics, snapshots = cu.train_and_monitor(
        ChamferLoss(),
        loss_name=chamfer_label,
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
    )
    all_results[chamfer_label] = (metrics, snapshots)

    # --- Power-SM3 τ=0.12 ---
    pm3_label = f'PM3 n={n_train}'
    print(f'\n=== Training {pm3_label} ===')
    metrics, snapshots = cu.train_and_monitor(
        PowerSoftMinLoss(temperature=0.12, power=3.0),
        loss_name=pm3_label,
        n_epochs=N_EPOCHS,
        snapshot_epochs=SNAPSHOT_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        tau_for_entropy=0.12,
    )
    all_results[pm3_label] = (metrics, snapshots)


=== Training Chamfer n=5000 ===
  [Chamfer n=5000] epoch   0 | loss=3.3090 dist=1.6213 div=1.450 H=0.230 gd=0.735 lr=3.00e-04
  [Chamfer n=5000] epoch   1 | loss=2.9035 dist=1.5384 div=1.551 H=0.198 gd=0.629 lr=3.00e-04
  [Chamfer n=5000] epoch   2 | loss=2.6684 dist=1.4232 div=1.672 H=0.206 gd=0.715 lr=3.00e-04
  [Chamfer n=5000] epoch   3 | loss=2.4255 dist=1.3397 div=1.756 H=0.210 gd=0.689 lr=3.00e-04
  [Chamfer n=5000] epoch   4 | loss=2.2376 dist=1.2754 div=1.827 H=0.198 gd=0.824 lr=3.00e-04
  [Chamfer n=5000] epoch   5 | loss=2.0890 dist=1.2362 div=1.850 H=0.203 gd=0.791 lr=3.00e-04
  [Chamfer n=5000] epoch   6 | loss=1.9803 dist=1.1987 div=1.889 H=0.204 gd=0.709 lr=3.00e-04
  [Chamfer n=5000] epoch   7 | loss=1.8965 dist=1.1886 div=1.894 H=0.208 gd=0.795 lr=3.00e-04
  [Chamfer n=5000] epoch   8 | loss=1.8307 dist=1.1704 div=1.915 H=0.210 gd=0.791 lr=3.00e-04
  [Chamfer n=5000] epoch   9 | loss=1.7638 dist=1.1594 div=1.938 H=0.207 gd=0.798 lr=3.00e-04
  [Chamfer n=5000] epoch  1

In [ ]:
cu.plot_monitoring(all_results)

In [ ]:
cu.summary_table(all_results)